# Write Li 2025 Figure 2-4 OD600 data as ATST

Seven readouts correspond to the seven OD600 panels in Figures 2, 3, and 4. 

In [1]:
from pathlib import Path
import datetime
import re
import pandas as pd
from openpyxl import load_workbook
from ATST import ATSTFile, MultiReadoutATST, validate_atst, write_multi_readout_atst
from ATST.blocks import Assay, Entities, EntityTable, FileMetadata, Layout, Metadata, Readings, ReadoutIds, Study

root = Path.cwd()
source = root / "sources/aem.02402-24-s0002.xlsx"
workbook = load_workbook(source, data_only=True, read_only=True)
# Workbook data rows end before the statistical comparison tables in each sheet.
# Figure 3E is an 8-hour endpoint: source rows 2-4 are replicate values.
panels = {
    "Fig_2A": ("Fig 2A", 8, "2", "CFS experiment; conditioned medium + 25% 4x LB", "low density", "OD600 trajectory"),
    "Fig_2B": ("Fig 2B", 11, "0.01", "synthetic 3OC12-HSL and C4-HSL", "low density", "OD600 trajectory"),
    "Fig_3A": ("Fig 3A", 8, "0.01", "QS mutants", "low density", "OD600 trajectory"),
    "Fig_3C": ("Fig 3C", 6, "0.01", "QS mutants", "high density", "OD600 trajectory"),
    "Fig_3E": ("Fig 3E", 4, "0.01", "25% 4x LB supplementation", "high density", "8-hour OD600 endpoint"),
    "Fig_4A": ("Fig 4A", 8, "0.01", "baicalein 100 uM", "low density", "OD600 trajectory"),
    "Fig_4C": ("Fig 4C", 8, "0.01", "baicalein 100 uM", "high density", "OD600 trajectory"),
}

In [2]:
file_info = FileMetadata(data={"file_name":"atst_li_2025.atst.txt","format":"ATST","format_version":"0.1","created_on":datetime.date.today().isoformat(),"field_delimiter":"TAB","encoding":"UTF-8"})
study = Study(data={"title":"Phage-host interaction in Pseudomonas aeruginosa clinical isolates with functional and altered quorum sensing systems","study_id":"doi:10.1128/aem.02402-24","authors":"Dandan Li; Na Li; Yu Chen; Yuxuan Yang; Jue Pan; Jiabing Lin; Xiaodong Gao; Rong Bao; Chunmei Zhou; Suzhen Wang; Bijie Hu; Demeng Tan","publication_date":"2025-04","publication_doi":"10.1128/aem.02402-24","description":"Source replicate OD600 measurements for Figures 2A/B, 3A/C/E, and 4A/C; phage phiPA2 and matched conditions where present."})
entities = Entities(tables={name:EntityTable(name="ENTITIES",table_name=name,pk=pk,data=pd.read_csv(root/"entities_definitions"/f"{name}.csv",dtype=str,keep_default_na=False)) for name,pk in [("ISOLATES","ISOLATE_ID"),("PHAGES","PHAGE_ID"),("MEDIA","MEDIUM_ID")]})

## Map source triplicates into seven readouts

The layouts retain `signaling_molecules` and `baicalein`; CFS values are omitted from layouts as requested and remain identifiable in the source files and curation notes. Excel statistical tables below the numeric blocks are excluded.

In [3]:
def condition(header):
    label = str(header).strip()
    host = label.split("+")[0].strip().replace(" ", "_").replace("Δ", "del_").replace("-", "_")
    infected = "phipa2" in label.lower()
    return host, infected, "3OC12-HSL 0.5 uM + C4-HSL 0.4 uM" if "3OC12-HSL" in label and "C4-HSL" in label else "", "100 uM" if "Baicalein" in label else ""

def curve_id(index):
    # Excel-style letters: A ... Z, AA ...
    result=""
    while index:
        index,rem=divmod(index-1,26)
        result=chr(65+rem)+result
    return "curve_"+result

readouts={}
for readout_id,(sheet_name,last_row,moi,context,density,kind) in panels.items():
    sheet=workbook[sheet_name]
    endpoint=(sheet_name=="Fig 3E")
    group_starts=[col for col in (range(1,sheet.max_column+1) if endpoint else range(2,sheet.max_column+1,3)) if sheet.cell(1,col).value is not None]
    times=[8] if endpoint else [sheet.cell(row,1).value for row in range(2,last_row+1)]
    readings=pd.DataFrame({"Time":times})
    layout_rows=[]
    for group,start in enumerate(group_starts):
        header=sheet.cell(1,start).value
        assert header, (sheet_name,start)
        host,infected,signaling,baicalein=condition(header)
        if endpoint: infected = True  # Fig. 3 caption and Methods specify phage addition.
        for replicate in range(1,4):
            cid=curve_id(len(layout_rows)+1)
            values=[sheet.cell(replicate+1,start).value] if endpoint else [sheet.cell(row,start+replicate-1).value for row in range(2,last_row+1)]
            assert len(values)==len(times) and all(isinstance(v,(int,float)) for v in values), (sheet_name,cid)
            readings[cid]=values
            layout_rows.append({"curve_id":cid,"type":"phage_exposed" if infected else "uninfected_control","isolate_id":host,"phage_id":"phiPA2" if infected else "","moi":moi if infected else "0","signaling_molecules":signaling,"baicalein":baicalein,"replicate":str(replicate)})
    layout=pd.DataFrame(layout_rows)
    assay=Assay(data={"readout_type":"absorbance","readout_unit":"od600","time_unit":"h","temperature_c":"37","medium_id":"LB","phage_inoculum_moi":moi,"culture_density":density,"experiment_context":context,"readout_description":kind,"measurement_method":"spectrophotometry","instrument":"SP-UV 200","replicates_per_condition":"3","source_reference":f"Paper Fig. {sheet_name[4:]}, caption; Methods, Phage-host interaction in liquid medium"})
    metadata=Metadata(data={"source_file":"read_shared_data/sources/aem.02402-24-s0002.xlsx","source_header_row":"1","source_data_rows":"2:4 (three replicate rows)" if endpoint else f"2:{last_row}","source_column_rule":"one condition per source column; rows 2-4 are replicates" if endpoint else "one condition per three adjacent source columns; left-to-right replicates","conversion":"numeric source values retained; statistical comparison tables omitted; endpoint time 8 h from Fig. 3 caption" if endpoint else "numeric source values and time retained; statistical comparison tables omitted"})
    readouts[readout_id]=ATSTFile(file_info=file_info,study=study,metadata=metadata,assay=assay,entities=entities,layout=Layout(data=layout),readings=Readings(data=readings))
readout_ids=ReadoutIds(data=pd.DataFrame({"readout_id":list(readouts)}))
for item in readouts.values():item.readout_ids=readout_ids
bundle=MultiReadoutATST(file_info,study,readout_ids,readouts,entities)
validate_atst(bundle)
pd.DataFrame([{"readout_id":k,"curves":len(v.layout.data),"time_points":len(v.readings.data)} for k,v in readouts.items()])

In [4]:
output=write_multi_readout_atst(bundle,root.parent/"atst_li_2025",linked_files=True)
print(output)
# Reuse identical metadata payloads across readouts even when assays or times differ.
manifest = output.read_text()
metadata_files = {}
for readout_id, item in bundle.readouts.items():
    signature = tuple(item.metadata.data.items())
    path = f"metadata/meta_{readout_id}.tsv"
    if signature in metadata_files:
        manifest = manifest.replace(f"file={path}", f"file={metadata_files[signature]}")
        (output.parent / path).unlink()
    else:
        metadata_files[signature] = path
output.write_text(manifest)


/Users/juanmantilla/Documents/TKI/ATST/examples/Public data /Li 2025/atst_li_2025/atst_li_2025.atst.txt
